# v14 PLM — Stage 2 training on vast.ai (run as-is, top to bottom)

Upload this single notebook to the vast.ai Jupyter instance and run all
cells. End result: `plm_weights.pt` ready to download.

**Before running — on your Mac:** commit & push everything the run needs:
```
git add CommunitySolutions/chronos_solver/v14 CommunitySolutions/chronos_solver/v13/*.json
git commit -m 'v14 PLM' && git push
```

If the GitHub repo is **private**, put a fine-grained personal access
token into `GIT_TOKEN` in the next cell. Public repo: leave it empty.

Training runs via `nohup`, so a dropped browser tab cannot kill it —
re-run the *monitor* cell anytime to see progress.

In [ ]:
# ---------------- Cell 1: clone the repo ----------------
import os
GIT_TOKEN = ""          # only needed if the repo is private
BRANCH    = "main"      # branch that contains the v14 folder

url = (f"https://{GIT_TOKEN}@github.com/shreyasmahimkar/arc-agi-3"
       if GIT_TOKEN else "https://github.com/shreyasmahimkar/arc-agi-3")
if not os.path.exists("/workspace/arc3"):
    !git clone --depth 1 --branch {BRANCH} {url} /workspace/arc3
else:
    !cd /workspace/arc3 && git pull
%cd /workspace/arc3
!ls CommunitySolutions/chronos_solver/v14

In [ ]:
# ---------------- Cell 2: dependencies ----------------
# torch ships with the vastai/pytorch image; only the two competition
# wheels + dotenv are needed on top.
!pip -q install \
    arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl \
    arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl \
    python-dotenv
print("deps installed")

In [ ]:
# ---------------- Cell 3: sanity — GPU + module smoke test ----------------
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
%cd /workspace/arc3/CommunitySolutions/chronos_solver/v14
!python -m plm.smoke    # must end with ALL SMOKE TESTS PASSED

In [ ]:
# ---------------- Cell 4: generate training data (~5 min) ----------------
!python gen_data.py --out /workspace/v14_shards \
    --episodes-per-game 400 --max-steps 150
!ls -lh /workspace/v14_shards | tail -5
!df -h /workspace | tail -1

In [ ]:
# ---------------- Cell 5: LAUNCH training (returns immediately) ----------------
# nohup-detached: survives tab drops and even Jupyter kernel restarts.
# Gates to hit: tokenizer pixel_acc >= 0.995, then HELDOUT_tok_acc >= 0.90.
!nohup python train_wm.py --phase all --shards /workspace/v14_shards \
    --epochs 20 --steps-per-epoch 1000 --bsz 256 \
    --holdout ls20,vc33,tu93,ft09,sp80 > /workspace/train.log 2>&1 &
import time; time.sleep(5)
!tail -3 /workspace/train.log

In [ ]:
# ---------------- Cell 6: MONITOR (re-run me anytime) ----------------
!ps aux | grep train_wm | grep -v grep || echo '*** TRAINING NOT RUNNING (finished or crashed - check log below) ***'
print('-' * 70)
!tail -15 /workspace/train.log
print('-' * 70)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# ---------------- Cell 7: VERIFY + stage for download (run when Cell 6
# shows training finished) ----------------
import torch, os
p = "/workspace/arc3/CommunitySolutions/chronos_solver/v14/plm_weights.pt"
state = torch.load(p, map_location="cpu", weights_only=True)
print("keys:", list(state))                      # expect tokenizer/belief/world_model
print(f"size: {os.path.getsize(p)/1e6:.1f} MB")
# copy weights + log to /workspace root for easy file-browser download
!cp {p} /workspace/plm_weights.pt && cp /workspace/train.log /workspace/train_final.log
print("\nDownload via the Jupyter file browser (/workspace):")
print("  plm_weights.pt  +  train_final.log")
print("Then DESTROY this instance on the vast.ai console.")

## After downloading

1. Put `plm_weights.pt` into your local `CommunitySolutions/chronos_solver/v14/`
   (`*.pt` is gitignored — it travels by hand, never via git).
2. Local gate: run the held-out-game eval vs the v13 bandit baseline.
3. Stage 3: update the `v14-plm` Kaggle dataset and submit
   (see DEPLOYMENT.md).